# Chess Engine with PyTorch

## Imports

In [1]:
import os
import numpy as np # type: ignore
import time
import torch
import torch.nn as nn # type: ignore
import torch.optim as optim # type: ignore
from torch.utils.data import DataLoader # type: ignore
from chess import pgn # type: ignore
from tqdm import tqdm # type: ignore

# Data preprocessing

## Load data

In [4]:
def load_games_with_limit(file_paths, max_games):
    all_games = []
    
    for file_path in tqdm(file_paths, desc="Processing Files"):
        with open(file_path, 'r') as pgn_file:
            while len(all_games) < max_games:
                game = pgn.read_game(pgn_file)
                
                if game is None:  # End of current file
                    break
                
                all_games.append(game)
                
        if len(all_games) >= max_games:
            print(f"\nReached limit of {max_games} games. Stopping.")
            break
            
    return all_games

# Setup paths
pgn_dir = "../../data/pgn"
files = [os.path.join(pgn_dir, f) for f in os.listdir(pgn_dir) if f.endswith(".pgn")]

# Define your strict limit
LIMIT_GAMES = 50000 

# Run the loader
games = load_games_with_limit(files, LIMIT_GAMES)
    

Processing Files:   0%|          | 0/23 [01:55<?, ?it/s]


Reached limit of 50000 games. Stopping.


## Convert data into tensors

In [5]:
from auxiliary_func import create_input_for_nn, encode_moves

In [6]:
X, y = create_input_for_nn(games)

print(f"NUMBER OF SAMPLES: {len(y)}")

NUMBER OF SAMPLES: 4424419


In [7]:
games = [] # clear up games array to save memory
X = X[0:2500000]
y = y[0:2500000]

In [8]:
y, move_to_int = encode_moves(y)
num_classes = len(move_to_int)

In [9]:
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)

# Preliminary actions

In [10]:
from dataset import ChessDataset
from model import ChessModel

In [11]:
# Create Dataset and DataLoader
dataset = ChessDataset(X, y)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Using device: {device}')

# Model Initialization
model = ChessModel(num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

Using device: cuda


# Training

In [12]:
num_epochs = 50
for epoch in range(num_epochs):
    start_time = time.time()
    model.train()
    running_loss = 0.0
    for inputs, labels in tqdm(dataloader):
        inputs, labels = inputs.to(device), labels.to(device)  # Move data to GPU
        optimizer.zero_grad()

        outputs = model(inputs)  # Raw logits

        # Compute loss
        loss = criterion(outputs, labels)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        running_loss += loss.item()
    end_time = time.time()
    epoch_time = end_time - start_time
    minutes: int = int(epoch_time // 60)
    seconds: int = int(epoch_time) - minutes * 60
    print(f'Epoch {epoch + 1 + 50}/{num_epochs + 1 + 50}, Loss: {running_loss / len(dataloader):.4f}, Time: {minutes}m{seconds}s')

100%|██████████| 39063/39063 [02:47<00:00, 233.54it/s]


Epoch 51/101, Loss: 3.6180, Time: 2m47s


100%|██████████| 39063/39063 [03:10<00:00, 204.67it/s]


Epoch 52/101, Loss: 2.7278, Time: 3m10s


100%|██████████| 39063/39063 [03:13<00:00, 202.02it/s]


...

Epoch 98/101, Loss: 1.5573, Time: 2m52s


100%|██████████| 39063/39063 [02:51<00:00, 228.14it/s]


Epoch 99/101, Loss: 1.5524, Time: 2m51s


100%|██████████| 39063/39063 [02:45<00:00, 235.58it/s]

Epoch 100/101, Loss: 1.5474, Time: 2m45s


# Save the model and mapping

In [13]:
# Save the model
torch.save(model.state_dict(), "../../models/TORCH_50EP_4M5SMPL_50KGM.pth")

In [14]:
import pickle

with open("../../models/heavy_move_to_int", "wb") as file:
    pickle.dump(move_to_int, file)